# 5.14 模型不動或震盪要怎麼找步幅？


參數w=1、目標位置3，代價(w−3)²。三種學習率0.01、0.1、2都沿初始下降方向走，是否都能降代價？讓每個候選回相同起點，再比較一步結果。

梯度2(w−3)=-4，新位置是w減學習率乘梯度。大步可能越過3，走到另一邊。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch

w = torch.tensor(1.0)
gradient = 2 * (w - 3)
print("起點代價", (w - 3).square().item())
for lr in [0.01, 0.1, 2.0]:
    new = w - lr * gradient
    print("步幅", lr, "新位置", new.item(), "新代價", (new - 3).square().item())

起點代價 4.0
步幅 0.01 新位置 1.0399999618530273 新代價 3.841600179672241
步幅 0.1 新位置 1.399999976158142 新代價 2.56000018119812
步幅 2.0 新位置 9.0 新代價 36.0


輸入起點與三種學習率，代碼每輪只計算new，沒有覆蓋w，因此比較條件一致。0.01到1.04，代價約3.8416，改善很少；0.1到1.4，代價2.56；2.0到9，代價36，遠比原來4差。大步雖沿正確初始方向，卻跨過目標3走上另一邊。這個具體反例比「大學習率不穩」一句話更能解釋發生什麼。

對真正模型可用小份固定資料做學習率範圍試驗，觀察前幾步loss是否下降、是否突然跳大、梯度有沒有異常。第一次先試量級，例如0.0001、0.001、0.01，再在有效範圍細分；不要只根據一次幸運下降就定案。各次試驗從同一組初始參數與optimizer歷史開始，使用同一資料順序及同樣更新步數，在同一步數觀察loss；最後參數與loss是要比較和記錄的結果，無需相同。種子是讓隨機選擇可重現的起始設定，應使用相同設定，見[5.15](https://birdhackor.github.io/tiny-perceptron-vlm/5.15.html)。

若觀察幾次更新後參數似乎完全沒變，先對照同一格更新前後的數值、保留足夠小數位，分清變化很小與完全沒有更新。接著確認是否真的呼叫 `optimizer.step()` 執行參數更新，回看[5.1](https://birdhackor.github.io/tiny-perceptron-vlm/5.1.html)；再檢查本批是否至少有一個答案位置參與代價計算，有效答案的計數與平均規則見[5.2](https://birdhackor.github.io/tiny-perceptron-vlm/5.2.html)。若出現無窮或 NaN（不是有效數值），要找出哪個輸入或運算先產生異常，並檢查所用分母是否為零。這個平方函數用0.5能一步到3，不能把算例中的0.1直接當語言模型最佳步幅。

練習只把起點w改成4，先手算梯度2、三種新位置為3.98、3.8、0，代價約0.9604、0.64、9，再執行核對。改起點會改梯度與絕對代價，但三個候選的新代價除以起點代價，仍分別約為0.9604、0.64、9；這個平方函數在兩個起點用0.5都能一步到3。因此這次練習沒有證明最佳步幅隨起點改變，只再次說明沿正確梯度方向走，仍不能保證任意大步幅有效。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
